# Estimación de la Función de Desempeño de Seguridad (FDS) para siniestralidad peatonal en Bogotá
### Modelo de conteo (Poisson / Binomial Negativa) a nivel de segmento vial

**Autor:** BRANSANT — Tesis de grado, Ciencia de Datos

---

## Índice

1. [Configuración del entorno y librerías](#1)
2. [Reproducibilidad — versiones de entorno](#2)
3. [Carga del conjunto de datos unificado](#3)
4. [Inspección del conjunto de datos](#4)
5. [Partición entrenamiento / validación](#5)
6. [Construcción de la variable de exposición (offset) y covariables](#6)
7. [Modelo Poisson](#7)
8. [Modelo Binomial Negativa](#8)
9. [Selección de modelo — prueba de razón de verosimilitud](#9)
10. [Comparación por Criterio de Información de Akaike (AIC)](#10)
11. [Evaluación en el conjunto de validación](#11)
12. [Resumen consolidado de resultados](#12)
13. [Resultados encontrados y observaciones críticas](#13)


<a id='1'></a>
## 1. Configuración del entorno y librerías

Importación de librerías de manejo de datos, geoespaciales y visualización, y configuración general de despliegue de tablas y gráficos.

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 80)
plt.rcParams['figure.figsize'] = (9, 6)


# Si es necesario DIR_PROC = 'Data/processed'
# os.makedirs(DIR_PROC, exist_ok=True)

<a id='2'></a>
## 2. Reproducibilidad — versiones de entorno

Captura de la versión de Python y de las librerías estadísticas empleadas (`statsmodels`, `scikit-learn`, `pandas`), con el fin de documentar el entorno de cómputo exacto en el que se generaron los resultados reportados en la metodología.

In [2]:
# Captura de version de Python y librerias clave para reproducibilidad --
# correr esto UNA VEZ en el entorno final antes de generar resultados definitivos,
# no en cualquier punto intermedio del desarrollo
import sys
import statsmodels
import sklearn
import pandas as pd

print(f"Python: {sys.version}")
print(f"statsmodels: {statsmodels.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
statsmodels: 0.14.6
scikit-learn: 1.9.0
pandas: 3.0.5


<a id='3'></a>
## 3. Carga del conjunto de datos unificado

Lectura del archivo `segmentos_unificado.csv`, resultado del preprocesamiento y unión espacial de la Malla Vial Integral con las capas de siniestralidad, semáforos, puentes y red peatonal.

In [3]:
df_unificado  = pd.read_csv('Data/processed/segmentos_unificado.csv')

<a id='4'></a>
## 4. Inspección del conjunto de datos

Vista general de las variables disponibles por segmento: longitud nativa, distancias a infraestructura peatonal y vehicular, conteo de siniestros peatonales, proxy poblacional y desagregación de severidad (heridos/muertos, no utilizada como variable dependiente).

> **Nota de calidad de dato a resolver:** en la muestra visible, `poblacion_localidad_media` aparece en 0.0 de forma sistemática. Verificar si corresponde a un valor real o a un fallo de unión/imputación antes de usar esta variable en el offset final.

In [4]:
df_unificado 

,long_seg_m,dist_puente_peatonal_m,dist_puente_vehicular_m,dist_semaforo_m,dist_peatonal_m,n_siniestros,poblacion_localidad_media,n_herido,n_muerto
0,30.958381,2149.708556,2248.410504,1410.063301,0.0,0,0.0,0,0
1,35.729601,2130.397897,2218.697332,1378.963885,0.0,0,0.0,0,0
2,31.900881,2108.777826,2173.881717,1232.553221,0.0,0,0.0,0,0
3,48.556662,2102.083822,2208.676848,1410.063301,0.0,0,0.0,0,0
4,21.075999,2106.174016,2164.688445,1255.390362,0.0,0,0.0,0,0
...,...,...,...,...,...,...,...,...,...
131699,138.327740,3199.597149,2074.655202,4829.450372,0.0,0,0.0,0,0
131700,47.720755,3310.416595,2202.181128,4938.129880,0.0,0,0.0,0,0
131701,131.444749,3349.305031,2246.587457,4976.154833,0.0,0,0.0,0,0
131702,204.070575,3349.305031,2246.587457,4976.154833,0.0,0,0.0,0,0


<a id='5'></a>
## 5. Partición entrenamiento / validación

Partición aleatoria simple (80% entrenamiento / 20% validación), consistente con el esquema de validación adoptado en la metodología (Vahedi Saheli y Effati, 2021). La semilla aleatoria queda fija y documentada para garantizar reproducibilidad.

In [5]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

# Semilla fija para reproducibilidad -- este numero debe quedar documentado
# textualmente en la metodologia, no solo en el codigo
SEMILLA = 1949

train, test = train_test_split(df_unificado, test_size=0.20, random_state=SEMILLA)
print(f"Entrenamiento: {len(train)} segmentos | Validacion: {len(test)} segmentos")

Entrenamiento: 105363 segmentos | Validacion: 26341 segmentos


<a id='6'></a>
## 6. Construcción de la variable de exposición (offset) y covariables

El offset combina la longitud nativa del segmento y el proxy poblacional de exposición peatonal, ambos en escala logarítmica, con coeficiente fijo igual a 1 (definición estructural de un offset en un modelo de conteo).

> **Pendiente de decisión:** la construcción del offset poblacional aquí implementada (`poblacion_localidad_media`) es un promedio a nivel de localidad, no la interpolación areal por buffer-intersección con manzanas DANE definida en la metodología. Esto reduce la variable a 20 valores posibles (uno por localidad) en lugar de un valor propio por segmento. Debe resolverse antes de reportar resultados finales.

> **Cobertura de covariables:** de la tabla de variables cerrada en la metodología, `MVI_VELREG`, `MVINUMC` y `ratio_cobertura_peatonal` no están incluidas en este ajuste; en su lugar se usa `dist_puente_vehicular_m`, no documentada previamente. Ajustar la tabla de variables o el modelo para que ambos coincidan.

In [6]:
import numpy as np

# OFFSET: aun no esta cerrado si combina longitud + poblacion o solo uno de los dos.
# Dejo la version combinada como placeholder -- AJUSTAR segun la decision final
train['offset'] = np.log(train['long_seg_m']) + np.log(train['poblacion_localidad_media'] + 1)
test['offset'] = np.log(test['long_seg_m']) + np.log(test['poblacion_localidad_media'] + 1)

covariables = ['dist_semaforo_m', 'dist_puente_peatonal_m', 'dist_puente_vehicular_m','dist_peatonal_m']

X_train = sm.add_constant(train[covariables])
X_test = sm.add_constant(test[covariables])
y_train = train['n_siniestros']
y_test = test['n_siniestros']

<a id='7'></a>
## 7. Modelo Poisson

Modelo de conteo con función de enlace logarítmica y offset de exposición. Sirve como modelo de referencia (caso límite de la Binomial Negativa cuando el parámetro de dispersión α = 0) para la prueba de selección de modelo de la sección 9.

In [7]:
modelo_poisson = sm.GLM(
    y_train, X_train,
    family=sm.families.Poisson(),
    offset=train['offset']
).fit()

print(modelo_poisson.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           n_siniestros   No. Observations:               105363
Model:                            GLM   Df Residuals:                   105358
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -22759.
Date:                Mon, 28 Sep 2026   Deviance:                       21633.
Time:                        03:39:17   Pearson chi2:                 9.56e+04
No. Iterations:                    15   Pseudo R-squ. (CS):           0.001863
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

<a id='8'></a>
## 8. Modelo Binomial Negativa

Modelo de conteo con parámetro de dispersión α estimado por máxima verosimilitud (`sm.NegativeBinomial`, no `sm.GLM`), necesario para que la prueba de razón de verosimilitud de la sección siguiente sea válida.

> **Advertencia de convergencia:** el ajuste generó múltiples `RuntimeWarning` (overflow, división por cero) y un `HessianInversionWarning` que impide calcular errores estándar confiables a partir de la matriz de covarianza. Es indicio de inestabilidad numérica, probablemente por covariables en escalas muy dispares (distancias en metros de orden 10¹–10³ sin estandarizar). Los p-valores reportados en el summary de este modelo deben tomarse con cautela hasta resolver esto.

In [8]:
# sm.NegativeBinomial (NO sm.GLM) estima alpha por MLE junto a los coeficientes,
# con error estandar propio -- esto es lo que permite la prueba de razon de
# verosimilitud descrita en la metodologia; GLM con family NB NO sirve para esto
modelo_bn = sm.NegativeBinomial(
    y_train, X_train,
    offset=train['offset'],
    loglike_method='nb2'  # varianza = mu + alpha*mu^2, la parametrizacion estandar en literatura de siniestros
).fit(disp=0)

print(modelo_bn.summary())
alpha_estimado = modelo_bn.params['alpha']
print(f"Alpha estimado: {alpha_estimado:.4f}")

c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3663: RuntimeWarning: overflow encountered in exp
  return np.exp(linpred)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3470: RuntimeWarning: overflow encountered in multiply
  dparams = exog*a1 * (y-mu)/(mu+a1)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3470: RuntimeWarning: invalid value encountered in multiply
  dparams = exog*a1 * (y-mu)/(mu+a1)
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\discrete\discrete_model.py:3470: Ru

                     NegativeBinomial Regression Results                      
Dep. Variable:           n_siniestros   No. Observations:               105363
Model:               NegativeBinomial   Df Residuals:                   105358
Method:                           MLE   Df Model:                            4
Date:                Mon, 28 Sep 2026   Pseudo R-squ.:                     nan
Time:                        03:41:51   Log-Likelihood:                -20843.
converged:                       True   LL-Null:                           nan
Covariance Type:            nonrobust   LLR p-value:                       nan
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     -16.5809      0.022   -740.457      0.000     -16.625     -16.537
dist_semaforo_m            -0.0005   3.53e-05    -13.554      0.000      -0.001      -0.000


c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
c:\Users\santiago\Desktop\Universidad\Tesis\Programacion\TT\.venv\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


<a id='9'></a>
## 9. Selección de modelo — prueba de razón de verosimilitud

Poisson es el caso límite de la Binomial Negativa cuando α = 0, por lo que ambos modelos están anidados y es válido aplicar una prueba de razón de verosimilitud. Dado que α = 0 se ubica en la frontera del espacio de parámetros (α ≥ 0), el p-valor se corrige a la mitad del que arrojaría una prueba χ² convencional (Cameron y Trivedi, 2013).

In [9]:
from scipy import stats

# Poisson es el caso limite de BN cuando alpha=0 -- por eso son modelos anidados
# y se puede aplicar LR test. PERO alpha=0 esta en la FRONTERA del espacio de
# parametros (alpha no puede ser negativo), por lo que la distribucion asintotica
# no es chi2(1) pura, sino una mezcla 50/50 de chi2(0) y chi2(1) (Cameron & Trivedi, 2013).
# El p-valor correcto es la MITAD del que arrojaria un chi2 test convencional.
LR_stat = 2 * (modelo_bn.llf - modelo_poisson.llf)
p_valor = 0.5 * (1 - stats.chi2.cdf(LR_stat, df=1))

print(f"Estadistico LR: {LR_stat:.4f}")
print(f"p-valor (corregido por frontera): {p_valor:.6f}")

if p_valor < 0.05:
    print("Alpha significativamente distinto de 0 -> se adopta Binomial Negativa")
    modelo_final = modelo_bn
else:
    print("No hay evidencia de sobredispersion significativa -> se conserva Poisson por parsimonia")
    modelo_final = modelo_poisson

Estadistico LR: 3831.2227
p-valor (corregido por frontera): 0.000000
Alpha significativamente distinto de 0 -> se adopta Binomial Negativa


<a id='10'></a>
## 10. Comparación por Criterio de Información de Akaike (AIC)

Criterio de contraste adicional a la prueba de razón de verosimilitud, tal como se especifica en la metodología.

In [10]:
print(f"AIC Poisson: {modelo_poisson.aic:.2f}")
print(f"AIC Binomial Negativa: {modelo_bn.aic:.2f}")

AIC Poisson: 45527.36
AIC Binomial Negativa: 41698.13


<a id='11'></a>
## 11. Evaluación en el conjunto de validación

Error Absoluto Medio (MAE) y Raíz del Error Cuadrático Medio (RMSE) del conteo predicho frente al observado en el 20% de validación, usando el modelo seleccionado en la sección 9.

> **Nota de interpretación:** dada la alta proporción de segmentos con conteo observado igual a cero, estas métricas están dominadas por aciertos triviales en ceros. Se recomienda complementar con un modelo nulo de referencia (solo offset, sin covariables) y con el gráfico de calibración por deciles de riesgo ya previsto en la metodología, antes de interpretar estos valores como evidencia de buen ajuste.

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Prediccion sobre el conjunto de validacion usando el offset de TEST, no de train
if modelo_final is modelo_bn:
    y_pred = modelo_final.predict(X_test, offset=test['offset'])
else:
    y_pred = modelo_final.predict(X_test, offset=test['offset'])

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE (validacion): {mae:.4f}")
print(f"RMSE (validacion): {rmse:.4f}")

MAE (validacion): 0.1660
RMSE (validacion): 0.9878


<a id='12'></a>
## 12. Resumen consolidado de resultados

Bloque de texto con los valores numéricos finales, listo para trasladar a la sección de resultados del documento de tesis.

In [12]:
# Genera el bloque de texto listo para pegar en la metodologia,
# con los valores reales en vez de placeholders -- revisar antes de copiar
reporte = f"""
N observaciones totales: {len(df_unificado)}
N entrenamiento: {len(train)} | N validacion: {len(test)}
Semilla: {SEMILLA}
Alpha BN estimado: {alpha_estimado:.4f}
LR statistic: {LR_stat:.4f} | p-valor: {p_valor:.6f}
AIC Poisson: {modelo_poisson.aic:.2f} | AIC BN: {modelo_bn.aic:.2f}
Modelo seleccionado: {'Binomial Negativa' if modelo_final is modelo_bn else 'Poisson'}
MAE validacion: {mae:.4f}
RMSE validacion: {rmse:.4f}
"""
print(reporte)


N observaciones totales: 131704
N entrenamiento: 105363 | N validacion: 26341
Semilla: 1949
Alpha BN estimado: 0.5217
LR statistic: 3831.2227 | p-valor: 0.000000
AIC Poisson: 45527.36 | AIC BN: 41698.13
Modelo seleccionado: Binomial Negativa
MAE validacion: 0.1660
RMSE validacion: 0.9878



<a id='13'></a>
## 13. Resultados encontrados y observaciones críticas

**Ajuste general:** sobre 131.704 segmentos (105.363 entrenamiento / 26.341 validación), la prueba de razón de verosimilitud rechaza contundentemente Poisson en favor de Binomial Negativa (LR = 3831.22, p < 0.001), confirmado por el AIC (Poisson: 45.527 vs. BN: 41.698). El parámetro de dispersión α = 0.522 (p < 0.001) confirma sobredispersión fuerte y significativa en el conteo de siniestros peatonales por segmento — resultado esperado dada la naturaleza del fenómeno y consistente con la literatura revisada.

**Variables significativas en el modelo seleccionado (BN):**
- `dist_semaforo_m`: coeficiente negativo y significativo (p < 0.001) — a menor distancia al semáforo más cercano, mayor riesgo esperado.
- `dist_puente_vehicular_m`: coeficiente negativo y significativo (p < 0.001).
- `dist_puente_peatonal_m`: **no significativo** en BN (p = 0.457), pese a aparecer significativo en Poisson (p = 0.007) — la significancia en Poisson es atribuible a errores estándar anti-conservadores por ignorar la sobredispersión ya confirmada; el resultado a reportar es el de BN.
- `dist_peatonal_m`: marginalmente no significativo en BN (p = 0.092).

**Desempeño fuera de muestra:** MAE = 0.166, RMSE = 0.988 sobre el conjunto de validación.

**Pendientes que condicionan la validez de estos resultados antes de reportarlos como definitivos:**
1. Resolver la construcción real del offset poblacional (interpolación areal por segmento, no promedio por localidad).
2. Incorporar `MVI_VELREG`, `MVINUMC` y `ratio_cobertura_peatonal` al modelo, o justificar explícitamente su exclusión.
3. Calcular manualmente el pseudo R² de McFadden (modelo nulo de solo offset vs. modelo completo), dado que el cálculo automático de statsmodels no lo produjo (`nan`).
4. Ajustar un modelo nulo de referencia y/o el gráfico de calibración por deciles antes de interpretar el MAE/RMSE como evidencia de buen desempeño, dada la alta proporción de ceros en la variable dependiente.